In [3]:
!pip install transformers sentencepiece scikit-learn -q

import pandas as pd
import numpy as np
import torch
import pickle
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


## Loead Data

In [4]:
BASE     = '/kaggle/input/datasets/sourabhsaxena/hatespeech01'
train_df = pd.read_csv(f'{BASE}/train.csv')
val_df   = pd.read_csv(f'{BASE}/val.csv')
test_df  = pd.read_csv(f'{BASE}/test.csv')

for df in [train_df, val_df, test_df]:
    df['text']  = df['text'].fillna('').astype(str)
    df['label'] = df['label'].astype(int)

# Combine train+val for baseline retraining
train_full = pd.concat([train_df, val_df], ignore_index=True)
print(f"Train: {len(train_full)} | Test: {len(test_df)}")

Train: 23626 | Test: 5907


## Train TF-IDF baseline:

In [5]:
# Cell 3 — Retrain TF-IDF + LR on full train set
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50000,
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

print("Training TF-IDF + LR...")
tfidf_pipeline.fit(train_full['text'], train_full['label'])

tfidf_preds = tfidf_pipeline.predict(test_df['text'])
tfidf_probs = tfidf_pipeline.predict_proba(test_df['text'])

print(f"TF-IDF Macro F1: {f1_score(test_df['label'], tfidf_preds, average='macro'):.4f}")

Training TF-IDF + LR...
TF-IDF Macro F1: 0.7118


## Get MuRIL probabilities:

In [6]:
# Cell 4 — MuRIL probabilities
HF_REPO   = 'sourabh5500/hate-speech-muril'
tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
model     = AutoModelForSequenceClassification.from_pretrained(HF_REPO)
model     = model.to(device)
model.eval()

def get_muril_probs(texts, batch_size=64):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(
            batch, max_length=128,
            padding=True, truncation=True,
            return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits, dim=1)
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_probs)

print("Getting MuRIL probabilities...")
muril_probs = get_muril_probs(test_df['text'].tolist())
print(f"✅ Done")

muril_preds = np.argmax(muril_probs, axis=1)
print(f"MuRIL Macro F1: {f1_score(test_df['label'], muril_preds, average='macro'):.4f}")

config.json:   0%|          | 0.00/815 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Invalid model-index. Not loading eval results into CardData.


model.safetensors:   0%|          | 0.00/950M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Getting MuRIL probabilities...
✅ Done
MuRIL Macro F1: 0.7293


## Ensemble combinations:

In [7]:
# Cell 5 — Try different ensemble strategies

y_true      = test_df['label'].values
muril_hof   = muril_probs[:, 1]
tfidf_hof   = tfidf_probs[:, 1]

strategies = {
    # Strategy 1: Simple average
    'Average (0.5+0.5)'      : (muril_hof * 0.5 + tfidf_hof * 0.5),
    # Strategy 2: MuRIL weighted
    'Weighted (0.7+0.3)'     : (muril_hof * 0.7 + tfidf_hof * 0.3),
    # Strategy 3: MuRIL heavy
    'Weighted (0.8+0.2)'     : (muril_hof * 0.8 + tfidf_hof * 0.2),
    # Strategy 4: OR logic (if either confident → HOF)
    'OR (muril>0.45|tfidf>0.55)': np.where(
        (muril_hof > 0.45) | (tfidf_hof > 0.55), 1, 0
    ).astype(float),
    # Strategy 5: AND logic (both must agree → HOF)
    'AND (muril>0.45&tfidf>0.45)': np.where(
        (muril_hof > 0.45) & (tfidf_hof > 0.45), 1, 0
    ).astype(float),
}

print("="*65)
print("ENSEMBLE STRATEGY COMPARISON")
print("="*65)
print(f"{'Strategy':<35} | {'Macro F1':>8} | {'HOF Recall':>10} | {'FN Rate':>8}")
print("-"*65)

# Baselines
print(f"{'MuRIL alone (t=0.5)':<35} | {f1_score(y_true, muril_preds, average='macro'):>8.4f} | "
      f"{f1_score(y_true, muril_preds, pos_label=1, average='binary'):>10.4f} | "
      f"{((muril_preds==0)&(y_true==1)).sum()/y_true.sum():>8.2%}")

tfidf_macro = f1_score(y_true, tfidf_preds, average='macro')
print(f"{'TF-IDF alone (t=0.5)':<35} | {tfidf_macro:>8.4f} | "
      f"{f1_score(y_true, tfidf_preds, pos_label=1, average='binary'):>10.4f} | "
      f"{((tfidf_preds==0)&(y_true==1)).sum()/y_true.sum():>8.2%}")

print("-"*65)

best_f1       = 0
best_strategy = ""
best_preds    = None

for name, probs_or_preds in strategies.items():
    if probs_or_preds.max() <= 1.0 and probs_or_preds.min() >= 0.0 and \
       set(np.unique(probs_or_preds)) != {0, 1}:
        # It's probabilities — apply threshold
        preds = (probs_or_preds >= 0.5).astype(int)
    else:
        preds = probs_or_preds.astype(int)

    macro  = f1_score(y_true, preds, average='macro')
    recall = f1_score(y_true, preds, pos_label=1, average='binary')
    fn     = ((preds==0) & (y_true==1)).sum() / y_true.sum()

    marker = " ← BEST" if macro > best_f1 else ""
    print(f"{name:<35} | {macro:>8.4f} | {recall:>10.4f} | {fn:>8.2%}{marker}")

    if macro > best_f1:
        best_f1       = macro
        best_strategy = name
        best_preds    = preds

print("="*65)
print(f"\n🏆 Best Strategy: {best_strategy}")
print(f"   Macro F1: {best_f1:.4f}")

ENSEMBLE STRATEGY COMPARISON
Strategy                            | Macro F1 | HOF Recall |  FN Rate
-----------------------------------------------------------------
MuRIL alone (t=0.5)                 |   0.7293 |     0.6974 |   33.76%
TF-IDF alone (t=0.5)                |   0.7118 |     0.6893 |   31.57%
-----------------------------------------------------------------
Average (0.5+0.5)                   |   0.7373 |     0.7134 |   30.19% ← BEST
Weighted (0.7+0.3)                  |   0.7380 |     0.7117 |   31.06% ← BEST
Weighted (0.8+0.2)                  |   0.7339 |     0.7057 |   32.08%
OR (muril>0.45|tfidf>0.55)          |   0.6943 |     0.7342 |   10.61%
AND (muril>0.45&tfidf>0.45)         |   0.7289 |     0.7080 |   29.64%

🏆 Best Strategy: Weighted (0.7+0.3)
   Macro F1: 0.7380


##  Final report on best ensemble:

In [8]:
# Cell 6 — Full report on best ensemble

print("="*60)
print(f"BEST ENSEMBLE — {best_strategy}")
print("="*60)
print(classification_report(
    y_true, best_preds,
    target_names=['NOT', 'HOF'],
    digits=4
))

fn = ((best_preds==0) & (y_true==1)).sum()
fp = ((best_preds==1) & (y_true==0)).sum()

print(f"FN (missed hate) : {fn} ({fn/y_true.sum():.2%})")
print(f"FP (over-flagged): {fp} ({fp/(y_true==0).sum():.2%})")

print("\n📊 IMPROVEMENT SUMMARY")
print("="*60)
print(f"{'Metric':<20} | {'MuRIL alone':>12} | {'Best Ensemble':>14}")
print("-"*50)

muril_f1  = f1_score(y_true, muril_preds, average='macro')
muril_fn  = ((muril_preds==0)&(y_true==1)).sum()/y_true.sum()
ens_fn    = fn/y_true.sum()
ens_f1    = best_f1

print(f"{'Macro F1':<20} | {muril_f1:>12.4f} | {ens_f1:>14.4f}")
print(f"{'FN Rate':<20} | {muril_fn:>12.2%} | {ens_fn:>14.2%}")
print(f"{'FN Reduction':<20} | {'':>12} | {(muril_fn-ens_fn)/muril_fn*100:>13.1f}%")

# Save results
pd.DataFrame([{
    'strategy'  : best_strategy,
    'macro_f1'  : best_f1,
    'muril_f1'  : muril_f1,
    'fn_rate'   : ens_fn,
    'muril_fn'  : muril_fn,
}]).to_csv('/kaggle/working/ensemble_results.csv', index=False)

print("\n✅ Saved ensemble_results.csv")

BEST ENSEMBLE — Weighted (0.7+0.3)
              precision    recall  f1-score   support

         NOT     0.7446    0.7851    0.7643      3164
         HOF     0.7355    0.6894    0.7117      2743

    accuracy                         0.7406      5907
   macro avg     0.7401    0.7372    0.7380      5907
weighted avg     0.7404    0.7406    0.7399      5907

FN (missed hate) : 852 (31.06%)
FP (over-flagged): 680 (21.49%)

📊 IMPROVEMENT SUMMARY
Metric               |  MuRIL alone |  Best Ensemble
--------------------------------------------------
Macro F1             |       0.7293 |         0.7380
FN Rate              |       33.76% |         31.06%
FN Reduction         |              |           8.0%

✅ Saved ensemble_results.csv
